<a href="https://colab.research.google.com/github/harikareddy1526/AI_DEV/blob/main/AI_Automation_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q -U fastapi httpx "transformers>=4.41,<5"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1 which is incompati

In [ ]:
from typing import Annotated

from fastapi import FastAPI
from pydantic import BaseModel, StringConstraints
from transformers import pipeline

# Load the model once.
analyser = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    framework="pt",
    device=-1,  # Use CPU.
)

app = FastAPI(title="AI Sentiment Automation")

# Reject empty, whitespace-only, and excessively long input.
Text = Annotated[
    str,
    StringConstraints(
        strip_whitespace=True,
        min_length=1,
        max_length=5000,
    ),
]


class SentimentRequest(BaseModel):
    text: Text


@app.get("/")
def home():
    return {"message": "Sentiment API is ready"}


@app.post("/analyse")
def analyse_sentiment(request: SentimentRequest):
    prediction = analyser(
        request.text,
        truncation=True,
        max_length=512,
    )[0]

    sentiment = prediction["label"]

    # Simple automation rule based on the AI prediction.
    action = (
        "Flag for customer support review"
        if sentiment == "NEGATIVE"
        else "Record as positive feedback"
    )

    return {
        "text": request.text,
        "sentiment": sentiment,
        "confidence": round(float(prediction["score"]), 4),
        "suggested_action": action,
    }

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
from fastapi.testclient import TestClient

with TestClient(app) as client:
    # Test 1: API health.
    response = client.get("/")
    assert response.status_code == 200

    # Test 2: Positive sentiment and its automation rule.
    response = client.post(
        "/analyse",
        json={"text": "I love this product! It is excellent."},
    )
    assert response.status_code == 200


    result = response.json()
    assert result["sentiment"] == "POSITIVE"
    assert 0 <= result["confidence"] <= 1
    assert result["suggested_action"] == "Record as positive feedback"
    print("Positive result:", result)

    # Test 3: Negative sentiment and its automation rule.
    response = client.post(
        "/analyse",
        json={"text": "This product is terrible. I hate it."},
    )
    assert response.status_code == 200

    result = response.json()
    assert result["sentiment"] == "NEGATIVE"
    assert result["suggested_action"] == "Flag for customer support review"
    print("Negative result:", result)

    # Test 4: Empty or whitespace-only input.
    for text in ["", "   "]:
        response = client.post("/analyse", json={"text": text})
        assert response.status_code == 422

    # Test 5: Missing required field.
    response = client.post("/analyse", json={})
    assert response.status_code == 422

print("\nAll tests passed!")

Positive result: {'text': 'I love this product! It is excellent.', 'sentiment': 'POSITIVE', 'confidence': 0.9999, 'suggested_action': 'Record as positive feedback'}
Negative result: {'text': 'This product is terrible. I hate it.', 'sentiment': 'NEGATIVE', 'confidence': 0.9998, 'suggested_action': 'Flag for customer support review'}

All tests passed!
